# Detección de Objetos y YOLO

La detección de objetos extiende la clasificación de imágenes de una manera crítica: un clasificador responde *¿qué hay en esta imagen?*, pero un detector también responde *¿dónde está?* — y lo hace para cada objeto presente, no solo el dominante.

Este notebook introduce los conceptos que hacen funcionar la detección de objetos, y luego muestra cómo usar YOLO para aplicarlos en la práctica. Entender los conceptos primero hace que la salida de YOLO sea mucho más fácil de razonar.

## Detección, Clasificación y Segmentación

A menudo se confunden tres tareas porque todas procesan imágenes:

| Tarea | Pregunta respondida | Salida |
|------|-------------------| -------|
| **Clasificación** | ¿Cuál es el objeto dominante? | Una etiqueta por imagen |
| **Detección** | ¿Qué objetos hay y dónde están? | Lista de etiquetas + recuadros delimitadores |
| **Segmentación** | ¿Qué píxeles pertenecen a cada objeto? | Máscaras de etiquetas por píxel |

La detección es más difícil que la clasificación porque el modelo debe predecir simultáneamente *si* existe un objeto, *qué* es y *dónde* está — para cada objeto, no solo uno.

La segmentación va aún más lejos: en lugar de un rectángulo, delinea la forma precisa de cada objeto. Verás un ejemplo de esto al final de este notebook cuando se demuestre la variante de segmentación de YOLO.

## Representación del Recuadro Delimitador

Un **recuadro delimitador** es un rectángulo que encierra un objeto en una imagen. Es la forma más común de representar la ubicación de un objeto porque es compacta (cuatro números) y fácil de dibujar y manipular.

Dos formatos aparecen repetidamente en los flujos de trabajo de detección:

1. **Formato de esquinas** `(x1, y1, x2, y2)`: esquina superior izquierda y esquina inferior derecha en coordenadas de píxeles. Natural para dibujar rectángulos.
2. **Formato centrado** `(x_center, y_center, width, height)`: punto central más dimensiones. Preferido para el entrenamiento — las salidas del modelo suelen parametrizarse como desplazamientos desde los centros de anclas.

En los archivos de etiquetas YOLO los valores también están **normalizados** al rango `[0, 1]` dividiendo por el ancho y alto de la imagen. La normalización hace las etiquetas independientes de la resolución: el mismo archivo de etiqueta funciona para una versión de 640×640 de una imagen y para una versión de 1280×1280.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

Las dos funciones siguientes convierten entre el formato de esquinas en píxeles y el formato centrado normalizado.
Una imagen mental útil es esta:
- El `formato de esquinas` describe el rectángulo por sus dos esquinas opuestas, por lo que es ideal para dibujar.
- El `formato YOLO` describe el mismo rectángulo por su punto central más su ancho y alto, por lo que es más fácil para que el modelo lo prediga.

Nada sobre el objeto cambia. Solo estamos cambiando el sistema de coordenadas usado para describir el mismo recuadro.

In [ ]:
def corner_to_yolo(box, img_width, img_height):
    """Convertir esquinas en píxeles (x1, y1, x2, y2) a (cx, cy, w, h) normalizado."""
    x1, y1, x2, y2 = box
    cx = ((x1 + x2) / 2) / img_width
    cy = ((y1 + y2) / 2) / img_height
    w  = (x2 - x1) / img_width
    h  = (y2 - y1) / img_height
    return cx, cy, w, h

def yolo_to_corner(box, img_width, img_height):
    """Convertir (cx, cy, w, h) normalizado a esquinas en píxeles (x1, y1, x2, y2)."""
    cx, cy, w, h = box
    x1 = (cx - w / 2) * img_width
    y1 = (cy - h / 2) * img_height
    x2 = (cx + w / 2) * img_width
    y2 = (cy + h / 2) * img_height
    return x1, y1, x2, y2

image_width, image_height = 640, 480
corner_box = (100, 50, 300, 200)
yolo_box   = corner_to_yolo(corner_box, image_width, image_height)
restored   = yolo_to_corner(yolo_box, image_width, image_height)

print(f'Formato de esquinas: {corner_box}')
print(f'Formato YOLO:        {[f"{v:.3f}" for v in yolo_box]}')
print(f'De vuelta a píxeles: {[round(v, 1) for v in restored]}')

Los valores normalizados siguen siendo válidos independientemente de la resolución de imagen usada en la inferencia, por eso los archivos de etiquetas YOLO los usan. La conversión de ida y vuelta confirma que no se pierde información al cambiar de formato.

## IoU: Intersección sobre Unión

Cuando un detector predice un recuadro, necesitamos una forma de medir qué tan cercano está a la anotación de verdad de tierra.

Piénsalo como comparar dos rectángulos para el **mismo objeto**:
- el **recuadro de verdad de tierra** fue dibujado por un anotador humano en el conjunto de datos etiquetado
- el **recuadro predicho** fue producido por el modelo en la inferencia

Si los dos rectángulos se superponen bien, la predicción es buena. Si apenas se superponen, la predicción es pobre incluso si la etiqueta de clase es correcta.

**Intersección sobre Unión (IoU)** compara el solapamiento entre dos rectángulos:

$$\text{IoU} = \frac{\text{Área de Intersección}}{\text{Área de Unión}}$$

- IoU = 1 → solapamiento perfecto (el recuadro predicho coincide exactamente con la verdad de tierra).
- IoU = 0 → sin solapamiento en absoluto.
- Cualquier valor entre medias indica qué tan cercana es la predicción.

IoU aparece en dos lugares muy diferentes en un pipeline de detección:
1. **Evaluación**: dado un recuadro predicho y uno de verdad de tierra, IoU decide si la predicción cuenta como verdadero positivo (el umbral suele ser 0.5).
2. **Posprocesamiento**: dados dos recuadros predichos con gran solapamiento, IoU decide si uno es un duplicado redundante que debe eliminarse — esto es la Supresión No Máxima.

In [ ]:
def calculate_iou(box1, box2):
    # El rectángulo de intersección está delimitado por las esquinas interiores de los dos recuadros.
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    # Si los recuadros no se superponen, una o ambas dimensiones de la intersección son negativas.
    # Limitar a cero da un área de intersección de cero en ese caso.
    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    # La unión cuenta cada píxel exactamente una vez: suma ambas áreas, resta el solapamiento.
    union = area1 + area2 - intersection
    return intersection / union if union > 0 else 0

box_a = (100, 100, 200, 200)
box_b = (150, 150, 250, 250)
print(f'IoU entre box_a y box_b: {calculate_iou(box_a, box_b):.3f}')

El gráfico siguiente hace visible la región de solapamiento.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.add_patch(patches.Rectangle((100,100), 100, 100, linewidth=2,
    edgecolor='steelblue', facecolor='steelblue', alpha=0.35, label='Box A (verdad de tierra)'))
ax.add_patch(patches.Rectangle((150,150), 100, 100, linewidth=2,
    edgecolor='tomato', facecolor='tomato', alpha=0.35, label='Box B (predicción)'))
ax.set_xlim(50, 300); ax.set_ylim(50, 300); ax.set_aspect('equal')
ax.legend(loc='upper left')
ax.set_title(f'IoU = {calculate_iou(box_a, box_b):.3f} — solo el área oscura de solapamiento cuenta')
plt.tight_layout(); plt.show()

## Supresión No Máxima (NMS)

Un detector no produce un solo recuadro por objeto. Produce muchos recuadros candidatos (uno por cada posición y escala que considera) y una puntuación de confianza para cada uno.

[![](../resources/images/nms.png)](https://www.analyticsvidhya.com/blog/2020/08/selecting-the-right-bounding-box-using-non-max-suppression-with-implementation/)

En la práctica esto significa que el mismo peatón puede producir una docena de recuadros superpuestos con coordenadas ligeramente distintas. Necesitamos una forma de quedarnos solo con el mejor.

**La Supresión No Máxima** resuelve esto con un algoritmo voraz:

1. Ordenar todos los recuadros restantes por puntuación de confianza (de mayor a menor).
2. Conservar el recuadro de mayor confianza.
3. Descartar todos los recuadros restantes cuya IoU con el recuadro conservado supere el umbral (son demasiado similares para ser objetos distintos).
4. Repetir desde el paso 1 con los recuadros supervivientes.

La intuición clave: alto solapamiento + menor confianza = detección redundante, no un segundo objeto. NMS no depende del modelo en sí; se aplica a la salida bruta para limpiarla.

[![](../resources/images/nms2.png)](https://blog.cubed.run/nms-non-maximum-suppression-157be5bc61ca)

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    # argsort devuelve los índices que ordenarían scores de menor a mayor; [::-1] invierte a mayor a menor.
    indices = np.argsort(scores)[::-1]
    keep = []
    while len(indices) > 0:
        current = indices[0]   # recuadro con mayor puntuación aún en consideración
        keep.append(current)
        if len(indices) == 1:
            break
        # Conservar solo los recuadros que no se solapan demasiado con el recuadro elegido.
        # Solapamiento por encima del umbral => probablemente el mismo objeto => suprimir.
        remaining = [i for i in indices[1:] if calculate_iou(boxes[current], boxes[i]) < iou_threshold]
        indices = remaining
    return keep

boxes = [
    (100, 100, 200, 200),          # detección más fuerte del objeto A
    (110, 105, 210, 205),          # duplicado de A (suprimido)
    (105, 102, 205, 202),          # otro duplicado de A (suprimido)
    (300, 300, 400, 400),          # objeto B, lejos (conservado)
]
scores = [0.9, 0.75, 0.8, 0.85]
kept = nms(boxes, scores, iou_threshold=0.5)
print(f'Antes de NMS: {len(boxes)} recuadros')
print(f'Después de NMS:  {len(kept)} recuadros  (índices: {kept})')

El cuarto recuadro (índice 3) está lejos y tiene una IoU esencialmente nula con los demás, por lo que sobrevive a NMS. El primer recuadro (índice 0) se conserva porque tiene la mayor confianza. Los recuadros 1 y 2 se solapan demasiado con el recuadro 0 y son suprimidos.

## Métricas de Detección

Tras ejecutar un detector sobre un conjunto de validación, necesitamos un número que resuma la calidad.

### Precisión y Recall

| Resultado | Significado |
|-----------|:------------|
| **Verdadero Positivo (TP)** | El recuadro predicho coincide con un recuadro de verdad de tierra (IoU > umbral) |
| **Falso Positivo (FP)** | El recuadro predicho no coincide con ninguna verdad de tierra — una **alucinación** |
| **Falso Negativo (FN)** | Un objeto real nunca fue detectado |

En detección, evaluamos **recuadros** predichos, no solo una etiqueta por imagen. Eso hace que los "verdaderos negativos" sean mal definidos (o abrumadoramente dominantes, dependiendo de cómo se cuenten las ubicaciones de fondo), por lo que la exactitud no es una métrica útil. En su lugar usamos:

$$\text{Precisión} = \frac{TP}{TP + FP}$$

*De todos los recuadros que el modelo predijo, ¿cuántos eran realmente correctos?*

$$\text{Recall} = \frac{TP}{TP + FN}$$

*De todos los objetos reales en la imagen, ¿cuántos encontró el modelo?*

Existe una tensión natural: aumentar el umbral de confianza eleva la precisión pero reduce el recall; reducirlo eleva el recall pero admite más falsos positivos.

### mAP

**mAP (mean Average Precision)** resume el compromiso precisión-recall sobre todos los umbrales de confianza y todas las clases. Dos variantes aparecen en los registros de entrenamiento de YOLO:

- **mAP50**: umbral de IoU fijo en 0.5 — considera una predicción correcta si el solapamiento de recuadros es ≥ 50%.
- **mAP50-95**: promediado sobre umbrales de 0.50 a 0.95 en pasos de 0.05 — una métrica más estricta que también penaliza la colocación imprecisa de recuadros.

En la práctica: si mAP50 es mucho mayor que mAP50-95, el modelo encuentra objetos pero no los localiza con precisión.

## Cómo Funciona YOLO

### La División entre Dos Etapas y Una Etapa

Los primeros detectores de aprendizaje profundo eran de **dos etapas**: primero proponen regiones candidatas y luego clasifican cada una. Faster R-CNN y sus predecesores son ejemplos. Suelen ser precisos, pero la estructura de dos pasadas añade latencia.

Los **detectores de una etapa** omiten el paso de propuesta y predicen recuadros y puntuaciones de clase directamente en un único pase hacia adelante por la red. YOLO, SSD y RetinaNet son ejemplos.

| Enfoque | Velocidad típica | Tendencia de precisión | Uso común |
|---------|-----------------|----------------------|-----------|
| Dos etapas | Más lento | A menudo mayor | Análisis sin tiempo real |
| Una etapa | Más rápido | Muy competitiva | Tiempo real, dispositivos de borde |

### Estructura Interna de YOLO

Los modelos YOLO modernos se componen de tres bloques funcionales:

1. **Backbone**: una CNN (o Vision Transformer) que extrae mapas de características a múltiples resoluciones espaciales de la imagen de entrada.
2. **Neck**: agrega características de distintas resoluciones para que el detector pueda manejar tanto objetos pequeños (necesitan detalle fino) como objetos grandes (necesitan contexto amplio). PAN (Path Aggregation Network) y FPN (Feature Pyramid Network) son diseños comunes de neck.
3. **Cabeza de detección**: para cada posición espacial y cada escala, predice desplazamientos de recuadro, puntuación de objetividad y probabilidades de clase.

El flujo de datos es:

```
imagen  →  backbone  →  mapas de características multiescala  →  neck  →  cabeza
                                                                              ↓
                             NMS  ←  candidatos brutos (miles de recuadros)
                              ↓
                        detecciones finales (típicamente < 100)
```

**Nota sobre los recuadros ancla**: las explicaciones antiguas de YOLO se construyen en torno a recuadros ancla fijos emparejados con objetos de verdad de tierra durante el entrenamiento. Ese concepto es útil para entender la historia, pero las versiones recientes de Ultralytics YOLO son en gran medida **libres de anclas** — la cabeza predice coordenadas de recuadro como desplazamientos directos sin formas de ancla predefinidas.

### Versiones de YOLO

| Versión | Año | Contribución clave |
|---------|-----|-------------------|
| YOLOv1 | 2016 | Detección en un solo pase; formulación basada en cuadrícula |
| YOLOv2/9000 | 2017 | Recuadros ancla; entrenamiento multiescala |
| YOLOv3 | 2018 | Predicciones de mapas de características multiescala |
| YOLOv4 | 2020 | Aumentación de datos potente; backbone compuesto |
| YOLOv5 | 2020 | Implementación limpia en PyTorch; fácil adopción |
| YOLOv8 | 2023 | API Ultralytics unificada; cabeza libre de anclas |
| YOLOv11 | 2024 | Refinamientos adicionales en el ecosistema Ultralytics |

## YOLO en la Práctica

El resto de este notebook usa la librería **Ultralytics**, que proporciona una API Python unificada para cargar modelos, ejecutar inferencia e inspeccionar resultados.

Instálala si es necesario:

```bash
pip install ultralytics   # o: uv add ultralytics
```

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO, settings
import cv2

# Almacenar los pesos del modelo descargados en una subcarpeta local en lugar del caché global.
MODELS_DIR = Path('../resources/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
settings.update({'weights_dir': str(MODELS_DIR.resolve())})

# Las salidas de inferencia (imágenes anotadas, etc.) van aquí.
OUTPUT_DIR = Path('../artifacts/outputs/yolo_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Cargar un Modelo Pre-entrenado

Un **modelo pre-entrenado** lleva pesos ya aprendidos a partir de un conjunto de datos etiquetado grande. No necesitamos entrenar nada — podemos usar estos pesos directamente para la inferencia.

Los modelos YOLO de Ultralytics están pre-entrenados en **COCO** (Common Objects in Context): un conjunto de datos de aproximadamente 330.000 imágenes que cubre 80 categorías de objetos cotidianos (personas, vehículos, animales, objetos del hogar y más). Esto significa que el modelo "conoce" esas 80 clases de serie — pero nada más allá de ellas, a menos que lo ajustemos finamente más adelante.

Las variantes de tamaño del modelo intercambian velocidad por precisión:

| Sufijo | Tamaño | Velocidad | Notas |
|--------|--------|-----------|-------|
| `n` | Nano | Más rápido | Buen punto de partida para enseñanza |
| `s` | Small | Rápido | |
| `m` | Medium | Moderado | |
| `l` | Large | Más lento | |
| `x` | Extra-grande | El más lento | Mayor precisión |

In [ ]:
# YOLO() descarga los pesos automáticamente en la primera ejecución.
model = YOLO(MODELS_DIR / 'yolov8n.pt')
print(f'Modelo: {model.model_name}')
print(f'Clases ({len(model.names)} en total de COCO):')
print({k: v for k, v in list(model.names.items())[:10]}, '...')

## Ejecutar Inferencia

Usamos `resources/images/bus.jpg` — una imagen de demo clásica de YOLO incluida en el repositorio. Muestra una escena callejera con un autobús y varios peatones, lo cual es una buena prueba para un modelo pre-entrenado en COCO ya que esas son clases de COCO.

In [ ]:
sample_path = Path('../resources/images/bus.jpg')
# Mostrar la imagen antes de la detección
img = cv2.imread(str(sample_path))
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Imagen de entrada antes de la detección')
plt.show()

`model()` acepta una ruta (o URL, array NumPy, etc.) y devuelve una lista de objetos `results` — uno por imagen de entrada. Cada objeto `results` contiene todos los recuadros predichos para esa imagen.

In [ ]:
results = model(sample_path, save=True, project=str(OUTPUT_DIR))
result  = results[0]  # solo se pasó una imagen
print(f'Detectados {len(result.boxes)} objetos')
for box in result.boxes:
    cls_id = int(box.cls[0])          # índice entero de clase
    conf   = float(box.conf[0])        # confianza 0–1
    xyxy   = box.xyxy[0].cpu().numpy() # esquinas en píxeles
    print(f'  {model.names[cls_id]:<12}  conf={conf:.2f}  box={xyxy.astype(int)}')

El método `result.plot()` dibuja todos los recuadros y etiquetas detectados directamente sobre la imagen.

In [ ]:
annotated = result.plot()
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title(f'{len(result.boxes)} detecciones')
plt.show()

## Formatos de Coordenadas

En la API de YOLO aparecen tres formatos de coordenadas. Elegir el correcto depende de lo que vayas a hacer a continuación:

| Atributo | Formato | Valores | Caso de uso |
|----------|---------|---------|-------------|
| `box.xyxy` | `(x1, y1, x2, y2)` | Píxeles | Dibujar recuadros, recortar regiones |
| `box.xywh` | `(cx, cy, w, h)` | Píxeles | Cálculos geométricos |
| `box.xywhn` | `(cx, cy, w, h)` | Normalizado 0–1 | Coincide con el formato de archivos de etiquetas YOLO |

Estos son los mismos tres formatos introducidos antes en el notebook — las funciones de conversión que escribimos son exactamente lo que la API hace internamente.

In [ ]:
for box in result.boxes:
    print('xyxy  (esquinas en píxeles):        ', box.xyxy[0].cpu().numpy())
    print('xywh  (centro + tamaño en píxeles):', box.xywh[0].cpu().numpy())
    print('xywhn (centro+tamaño normalizado): ', box.xywhn[0].cpu().numpy())
    break  # una detección es suficiente para ilustrar

## Filtrar Detecciones

Los parámetros de inferencia permiten controlar qué detecciones se devuelven. Los valores por defecto son `conf=0.25` e `iou=0.45`.

- `conf` — puntuación de confianza mínima; las detecciones por debajo de este valor se descartan silenciosamente. Subirlo intercambia recall por precisión: menos resultados, pero más fiables.
- `classes` — lista blanca de IDs de clase; otras clases se ignoran incluso con alta confianza. Los índices de clase siguen el orden de COCO (0 = person, 2 = car, 5 = bus, …).
- `iou` — umbral de IoU usado dentro de NMS. Subirlo hace NMS más permisivo: dos recuadros deben solaparse en *más* de esta fracción antes de que uno sea suprimido. Esto puede preservar objetos genuinamente cercanos que el umbral por defecto suprimiría erróneamente. Nota: NMS de Ultralytics es agnóstico a la clase por defecto, por lo que un `iou` bajo puede suprimir detecciones de distintas clases cuando sus recuadros se solapan (por ejemplo, un recuadro de persona dentro de uno de autobús).

In [ ]:
def labels(result):
    return [model.names[int(b.cls[0])] for b in result.boxes]

# Línea base: parámetros por defecto (conf=0.25, iou=0.45)
results_base = model(sample_path, verbose=False)
print(f'línea base     {len(results_base[0].boxes)} detecciones — {labels(results_base[0])}')

# Subir conf a 0.5 elimina las detecciones de baja confianza
results_conf = model(sample_path, conf=0.5, verbose=False)
print(f'conf=0.5       {len(results_conf[0].boxes)} detecciones — {labels(results_conf[0])}')

# Lista blanca: solo personas (0) y coches (2); el autobús (clase 5) se ignora sin importar la confianza
results_cls = model(sample_path, classes=[0, 2], verbose=False)
print(f'classes=[0,2]  {len(results_cls[0].boxes)} detecciones — {labels(results_cls[0])}')

# NMS permisivo (iou=0.7): los recuadros deben solaparse en > 70% antes de que uno sea suprimido.
# Útil cuando objetos densamente agrupados de la misma clase están siendo fusionados erróneamente.
results_iou = model(sample_path, iou=0.7, verbose=False)
print(f'iou=0.7        {len(results_iou[0].boxes)} detecciones — {labels(results_iou[0])}')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 7))
fig.subplots_adjust(wspace=0.02)
configs = [
    (results_base[0], 'línea base\nconf=0.25, iou=0.45'),
    (results_conf[0], 'conf=0.5\nelimina recuadros de baja confianza'),
    (results_cls[0],  'classes=[0,2]\nautobús (clase 5) excluido'),
    (results_iou[0],  'iou=0.7\nNMS permisivo'),
]
for ax, (r, title) in zip(axes, configs):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    ax.set_title(f'{title}\n({len(r.boxes)} detecciones)', fontsize=9)
    ax.axis('off')
plt.show()

## Inferencia en Vídeo

YOLO procesa vídeo como una secuencia de fotogramas. El único cambio práctico respecto a la inferencia en imagen es el flag `stream=True`, que convierte la llamada en un generador en lugar de recopilar todos los resultados en memoria a la vez. Esto es importante para vídeos largos.

La relación entre la velocidad del modelo y la inferencia en vídeo es directa: un modelo que tarda 50 ms por fotograma admite como máximo 20 FPS. El tamaño del modelo, la resolución de entrada, el hardware y el tamaño del batch afectan a la latencia. Ponemos esto en perspectiva en la sección de comparación de modelos más adelante.

El código descarga un vídeo corto de muestra para la demo. Los vídeos no se incluyen en el repositorio por su tamaño de archivo, pero la descarga se ejecuta una vez y se reutiliza.

In [ ]:
import urllib.request
video_url  = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/decelera_portrait_min.mov'
video_path = OUTPUT_DIR / 'sample_video.mov'
if not video_path.exists():
    try:
        urllib.request.urlretrieve(video_url, str(video_path))
        print(f'Descargado en {video_path}')
    except Exception as e:
        print(f'Descarga fallida: {e}')
        video_path = None

In [ ]:
if video_path and video_path.exists():
    # stream=True: decodificar y procesar fotogramas uno a la vez en lugar de cargar
    # todo el vídeo en memoria. Esencial para vídeos largos.
    results_video = model(str(video_path), stream=True, save=True, project=str(OUTPUT_DIR))
    frame_count = 0
    for r in results_video:
        frame_count += 1
        if frame_count >= 5:
            break
    print(f'Procesados {frame_count} fotogramas (demo limitada a 5)')

## Segmentación y Otras Tareas

Ultralytics ofrece variantes de modelo específicas por tarea usando la misma API. Los modelos de segmentación producen máscaras por píxel además de recuadros delimitadores, respondiendo a la pregunta más precisa de *¿qué píxeles pertenecen a este objeto?* en lugar de solo *¿dónde está el recuadro?*

Se puede reconocer qué tarea realiza un modelo por el sufijo de su nombre de archivo:

| Nombre de archivo | Tarea |
|-------------------|-------|
| `yolov8n.pt` | Detección (solo recuadros) |
| `yolov8n-seg.pt` | Segmentación de instancias |
| `yolov8n-pose.pt` | Estimación de pose humana |
| `yolov8n-cls.pt` | Clasificación de imágenes |

In [ ]:
seg_model   = YOLO(MODELS_DIR / 'yolov8n-seg.pt')
seg_results = seg_model(sample_path, save=True, project=str(OUTPUT_DIR))
if seg_results[0].masks is not None:
    print(f'Forma del tensor de máscaras: {seg_results[0].masks.data.shape}')
    print('Primera dimensión = número de objetos detectados')
    print('Dimensiones restantes = resolución de la máscara (puede diferir del tamaño de la imagen de entrada)')

In [ ]:
seg_annotated = seg_results[0].plot()
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(seg_annotated, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Segmentación de instancias: cada objeto recibe su propia máscara de color')
plt.show()

## Comparación de Modelos: Velocidad vs Precisión

Elegir un tamaño de modelo es un compromiso práctico. El benchmark siguiente cronometra 10 inferencias consecutivas tras una ejecución de calentamiento (la primera inferencia suele ser más lenta por la compilación de kernels de GPU o la asignación de memoria, por lo que se excluye de la medición).

In [ ]:
import time
for model_name in ['yolov8n.pt', 'yolov8s.pt']:
    m = YOLO(MODELS_DIR / model_name)
    m(sample_path, verbose=False)  # calentamiento
    t0 = time.time()
    for _ in range(10):
        m(sample_path, verbose=False)
    ms_per_image = (time.time() - t0) / 10 * 1000
    print(f'{model_name:20}  {ms_per_image:.1f} ms/imagen  ({1000/ms_per_image:.0f} FPS teórico)')

Estos números dependen en gran medida del hardware. En una CPU la diferencia entre nano y small puede ser de 2–3×; en una GPU se reduce considerablemente. Para decisiones de despliegue, mide en el dispositivo objetivo.

## Exportar para Despliegue

Un modelo entrenado o ajustado finamente en Python puede exportarse a un formato que funcione en un entorno diferente. Cada objetivo de exportación está pensado para distintos escenarios de despliegue:

| Formato | Caso de uso |
|---------|-------------|
| `onnx` | Multiplataforma; compatible con ONNX Runtime, TensorRT, OpenCV DNN, … |
| `torchscript` | Permanecer en el ecosistema PyTorch sin intérprete Python |
| `coreml` | macOS e iOS (Apple Silicon o Neural Engine) |
| `tflite` | Android y microcontroladores |
| `engine` | NVIDIA TensorRT para máximo rendimiento GPU |

La elección depende enteramente de dónde se desplegará el modelo, no del framework de entrenamiento.

In [ ]:
# Descomentar para exportar. El archivo exportado aparece junto a los pesos.
# model.export(format='onnx')
export_info = [
    ('onnx',        'Multiplataforma, CPU/GPU'),
    ('torchscript', 'Despliegue PyTorch (sin Python)'),
    ('coreml',      'macOS / iOS'),
    ('tflite',      'Android / Dispositivos de borde'),
    ('engine',      'NVIDIA TensorRT'),
]
print('Formatos de exportación disponibles:')
for fmt, desc in export_info:
    print(f'  {fmt:<14} {desc}')

## Resumen

| Concepto | Punto clave |
|----------|-------------|
| **Recuadro delimitador** | Cuatro números (formato de esquinas o centrado) que localizan un objeto |
| **IoU** | Ratio de solapamiento; usado en evaluación (decisión TP/FP) y en NMS (eliminación de duplicados) |
| **NMS** | Algoritmo voraz que elimina duplicados de menor confianza |
| **mAP50 / mAP50-95** | Métricas estándar; mAP50-95 también penaliza la colocación imprecisa de recuadros |
| **YOLO** | Detector de una etapa; rápido, práctico, una llamada API para imágenes y vídeo |
| **Formatos de coordenadas** | xyxy para dibujar, xywh para geometría, xywhn para archivos de etiquetas |
| **Variantes de modelo** | `n` → `s` → `m` → `l` → `x`: más grande es más lento pero más preciso |